# chk1 数据、训练与模型审计

本 notebook 只读地复算总结报告中的关键数字。它不加载模型权重、不访问 DeepSeek，也不修改任何数据或训练产物。

In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import json

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo = next(path for path in candidates if (path / 'configs/retrain_v2/dag.yaml').is_file())
audit_path = repo / 'docs/summary/20260805T124140Z/chk1_audit.py'
spec = spec_from_file_location('chk1_audit', audit_path)
audit = module_from_spec(spec)
spec.loader.exec_module(audit)
results = audit.run_audit(repo)
print(json.dumps(results, ensure_ascii=False, indent=2, sort_keys=True))

{
  "base_release": {
    "artifact_sha256": "b8128b394420f90d8e7cb2e0e422a7dc097b5d6c68adb30f73ded29f764c040f",
    "audit_statuses": {
      "encoding": "passed",
      "point_in_time": "passed",
      "reference_leakage": "passed",
      "schema": "passed",
      "split_integrity": "passed",
      "target_consistency": "passed",
      "teacher_grounding": "passed",
      "token_budget": "passed"
    },
    "empty_final_responses": 0,
    "grpo_counts": {
      "eval": 199,
      "test": 190,
      "train": 493
    },
    "missing_reasoning_boundary": 0,
    "release_id": "analysis_base_full_v7_automated_v3_20260804",
    "sft_counts": {
      "eval": 199,
      "test": 190,
      "train": 1355
    },
    "token_budget": {
      "budgets": {
        "completion": null,
        "prompt": 3072,
        "total": 7168
      },
      "max_observed": {
        "completion": 4464,
        "prompt": 2735,
        "total": 6855
      },
      "split_counts": {
        "eval": 199,
        "te

## 关键一致性检查

下面的断言解释 canonical 训练集 1683 与最终 SFT 训练集 1355 的差异，并确认模型已经封存且被下游运行复用。

In [2]:
roles = results['base_release']['train_roles']
assert roles['sft_only'] + roles['grpo_only'] + roles['shared'] == 1683
assert roles['sft_only'] + roles['shared'] == results['base_release']['sft_counts']['train'] == 1355
assert roles['grpo_only'] + roles['shared'] == results['base_release']['grpo_counts']['train'] == 493
assert results['training']['status'] == 'sealed'
assert all(item['source_artifact_sha256'] == results['training']['merged_sha256'] for item in results['downstream_reuse'])
print('PASS: split accounting, sealed artifact, and downstream imports are consistent.')

PASS: split accounting, sealed artifact, and downstream imports are consistent.


In [3]:
summary = {
    'DeepSeek retrieval rate': results['generation']['retrieval_rate'],
    'canonical admitted rows': results['canonical_release']['accepted'],
    'SFT train/validation/test': results['base_release']['sft_counts'],
    'final eval loss': results['training']['eval_loss'],
    'epoch-2 eval token accuracy': results['training']['eval_history'][-1]['eval_mean_token_accuracy'],
    'merged artifact SHA-256': results['training']['merged_sha256'],
}
summary

{'DeepSeek retrieval rate': 0.9990552668871044,
 'canonical admitted rows': 2072,
 'SFT train/validation/test': {'train': 1355, 'eval': 199, 'test': 190},
 'final eval loss': 0.7877293825149536,
 'epoch-2 eval token accuracy': 0.7944194704294205,
 'merged artifact SHA-256': '9b355f903b7722f274bca1bc226bf75c1da4ebe449de726174bd1d054ff8948c'}